# Pre-Processing Visualization: Bathymetry, Wave & Wind Climate

**Authors:** Alireza Shadmani, Mohammad Reza Nikoo, Amir H. Gandomi

---

## Overview

This notebook produces **publication-quality maps** for two case-study regions used throughout the textbook:

1. **California Bay** (San Francisco coastal area)
2. **Oman Coast** (Arabian Sea / Gulf of Oman)

For each region we visualize:

| Layer | Data Source | What it shows |
|-------|-----------|---------------|
| Bathymetry | GEBCO ASCII grid | Seafloor depth contours with land masking |
| Wave climate | ERA5 reanalysis (NetCDF) | Mean significant wave height ($H_s$), mean wave direction, and wave power density |
| Wind climate | ERA5 reanalysis (NetCDF) | Mean wind speed with vector overlay, plus a wind-rose histogram |
| Time series | ERA5 wave data | Temporal evolution of domain-averaged $H_s$ and $T_p$ with rolling means |

### Why pre-processing matters

Before running any numerical wave model (e.g. SWAN), it is essential to **inspect and understand the input data**. Pre-processing visualizations help you:

- Verify that bathymetry grids are correctly oriented and free of artefacts
- Confirm that wave/wind forcing fields cover the model domain
- Identify dominant wave directions, seasonal patterns, and spatial gradients
- Detect data gaps or unrealistic values early in the workflow

### Dependencies

```
numpy, scipy, netCDF4, matplotlib
```

## 1. Imports and Configuration

We begin by importing the required libraries and setting up file paths for both study regions. The `BASE` path points to the parent directory containing the raw data folders (`california_bay.F/` and `Oman/`), while `OUT` is where generated figures will be saved.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import netCDF4 as nc
from scipy.interpolate import RectBivariateSpline

%matplotlib inline
plt.rcParams.update({'figure.max_open_warning': 0})

In [ ]:
# ---------------------------------------------------------------------------
# Paths — adjust BASE if your data folders are in a different location
# ---------------------------------------------------------------------------
BASE = os.path.join(os.path.dirname(os.getcwd()), '..')
OUT  = os.getcwd()   # figures saved alongside this notebook

REGIONS = {
    'California Bay': {
        'gebco': os.path.join(BASE, 'california_bay.F', 'GEBCO.asc'),
        'wave':  os.path.join(BASE, 'california_bay.F', 'WAVE.nc'),
        'wind':  os.path.join(BASE, 'california_bay.F', 'WIND.nc'),
        'tag':   'california',
    },
    'Oman Coast': {
        'gebco': os.path.join(BASE, 'Oman', 'gebco.asc'),
        'wave':  os.path.join(BASE, 'Oman', 'Wave-2021-April-May.nc'),
        'wind':  os.path.join(BASE, 'Oman', 'Wind-2021-April-May.nc'),
        'tag':   'oman',
    },
}

print("Regions configured:")
for name, paths in REGIONS.items():
    print(f"  {name}:")
    for k, v in paths.items():
        if k != 'tag':
            exists = os.path.exists(v)
            print(f"    {k}: {'OK' if exists else 'MISSING'} — {os.path.basename(v)}")

## 2. Data Loaders

Three helper functions load the raw input data:

### 2.1 GEBCO Bathymetry (ASCII Grid)

The [GEBCO](https://www.gebco.net/) gridded bathymetry is distributed as an ASCII raster (`.asc`) with a 6-line header specifying the grid dimensions, origin, cell size, and no-data value. The function reads the header, loads the elevation matrix, and returns 1-D longitude/latitude vectors plus the 2-D elevation array (negative = below sea level).

In [ ]:
def load_gebco(path):
    """Load GEBCO ASCII-grid bathymetry."""
    with open(path) as f:
        ncols = int(f.readline().split()[1])
        nrows = int(f.readline().split()[1])
        xll   = float(f.readline().split()[1])
        yll   = float(f.readline().split()[1])
        cs    = float(f.readline().split()[1])
        nd    = float(f.readline().split()[1])
    elev = np.loadtxt(path, skiprows=6)
    elev[elev == nd] = np.nan
    lons = xll + (np.arange(ncols) + 0.5) * cs
    lats = yll + (nrows - 1 - np.arange(nrows) + 0.5) * cs
    return lons, lats, elev

### 2.2 ERA5 Wave Data (NetCDF)

ERA5 wave reanalysis files contain hourly fields of:
- `swh` — Significant wave height $H_s$ (m)
- `pp1d` — Peak wave period $T_p$ (s)
- `mwd` — Mean wave direction (degrees, nautical convention)

We compute:
- **Time-averaged** $\overline{H_s}$, $\overline{T_p}$
- **Circular mean direction** using the arctangent of averaged sine/cosine components
- **Wave power density** $P = \frac{\rho g^2}{64\pi} H_s^2 T_p$ (kW/m), the energy flux per metre of wave crest

In [ ]:
def load_wave(path):
    """Load ERA5 wave NetCDF -> dict with time-averaged and raw fields."""
    ds = nc.Dataset(path)
    lat = ds.variables['latitude'][:]
    lon = ds.variables['longitude'][:]
    swh = ds.variables['swh'][:]       # (t, lat, lon) Hs
    pp1d = ds.variables['pp1d'][:]     # Tp
    mwd = ds.variables['mwd'][:]       # mean wave direction (deg)

    # Parse timestamps
    tvar = ds.variables['valid_time']
    cal = tvar.calendar if hasattr(tvar, 'calendar') else 'standard'
    times = nc.num2date(tvar[:], tvar.units, cal)
    ds.close()

    # Time-averages
    mean_hs = np.ma.mean(swh, axis=0)
    mean_tp = np.ma.mean(pp1d, axis=0)

    # Circular mean direction
    mwd_rad = np.deg2rad(mwd)
    mean_dir = np.rad2deg(np.arctan2(
        np.ma.mean(np.sin(mwd_rad), axis=0),
        np.ma.mean(np.cos(mwd_rad), axis=0),
    )) % 360

    # Wave power density (kW/m)
    rho_g2 = 1025 * 9.81**2 / (64 * np.pi)
    P = rho_g2 * swh**2 * pp1d / 1000
    mean_P = np.ma.mean(P, axis=0)

    return dict(lat=lat, lon=lon, mean_hs=mean_hs, mean_tp=mean_tp,
                mean_dir=mean_dir, mean_P=mean_P,
                swh=swh, pp1d=pp1d, mwd=mwd, times=times)

### 2.3 ERA5 Wind Data (NetCDF)

ERA5 10-metre wind components `u10` (eastward) and `v10` (northward) are loaded and converted to:
- **Wind speed**: $\sqrt{u_{10}^2 + v_{10}^2}$
- **Wind direction** (meteorological "from" convention): $\arctan(-u_{10},\, -v_{10})$

In [ ]:
def load_wind(path):
    """Load ERA5 wind NetCDF -> dict with time-averaged fields."""
    ds = nc.Dataset(path)
    lat = ds.variables['latitude'][:]
    lon = ds.variables['longitude'][:]
    u10 = ds.variables['u10'][:]
    v10 = ds.variables['v10'][:]
    ds.close()

    speed = np.sqrt(u10**2 + v10**2)
    direction = (np.rad2deg(np.arctan2(-u10, -v10)) + 360) % 360  # "from" direction

    mean_speed = np.ma.mean(speed, axis=0)
    mean_u = np.ma.mean(u10, axis=0)
    mean_v = np.ma.mean(v10, axis=0)

    return dict(lat=lat, lon=lon, mean_speed=mean_speed,
                mean_u=mean_u, mean_v=mean_v,
                speed=speed, direction=direction)

## 3. Interpolation Helper: Upscale and Land-Mask

ERA5 fields have a coarse spatial resolution (~0.25 deg). For publication-quality maps we **bi-cubic interpolate** these onto a finer grid (~0.02 deg, roughly matching GEBCO resolution) and then **mask land cells** using the GEBCO elevation data.

Key steps:
1. Fill masked values so the spline doesn't blow up
2. Build a `RectBivariateSpline` (SciPy) on the coarse grid
3. Evaluate on the fine grid
4. Use GEBCO elevation >= 0 to flag land pixels as `NaN`

In [ ]:
def upscale_and_mask(coarse_lon, coarse_lat, field, gebco, target_res=0.02):
    """
    Smooth a coarse ERA5 field onto a finer regular grid and mask land.

    Parameters
    ----------
    coarse_lon, coarse_lat : 1-D arrays (ERA5 coordinates).
    field        : 2-D array on the coarse grid.
    gebco        : (lons, lats, elev) tuple from load_gebco().
    target_res   : output grid spacing in degrees (~2 km).

    Returns
    -------
    fine_lon, fine_lat, fine_field  (all 2-D meshgrids / masked array)
    """
    arr = np.ma.filled(field, np.nan)

    # Latitude must be ascending for RectBivariateSpline
    lat_asc = coarse_lat if coarse_lat[0] < coarse_lat[-1] else coarse_lat[::-1]
    dat = arr if coarse_lat[0] < coarse_lat[-1] else arr[::-1, :]

    # Replace remaining NaNs with column means for spline stability
    for j in range(dat.shape[1]):
        col = dat[:, j]
        nans = np.isnan(col)
        if nans.all():
            dat[:, j] = 0.0
        elif nans.any():
            dat[nans, j] = np.nanmean(col)

    spline = RectBivariateSpline(lat_asc, coarse_lon, dat, kx=3, ky=3)

    fine_lat_1d = np.arange(lat_asc[0], lat_asc[-1] + 1e-9, target_res)
    fine_lon_1d = np.arange(coarse_lon[0], coarse_lon[-1] + 1e-9, target_res)
    fine_vals = spline(fine_lat_1d, fine_lon_1d)

    FLON, FLAT = np.meshgrid(fine_lon_1d, fine_lat_1d)

    # --- Land mask from GEBCO ---
    g_lon, g_lat, g_elev = gebco
    lat_idx = np.searchsorted(g_lat[::-1], fine_lat_1d)
    lat_idx = np.clip(len(g_lat) - 1 - lat_idx, 0, len(g_lat) - 1)
    lon_idx = np.searchsorted(g_lon, fine_lon_1d)
    lon_idx = np.clip(lon_idx, 0, len(g_lon) - 1)

    li, lj = np.meshgrid(lat_idx, lon_idx, indexing='ij')
    land = g_elev[li, lj] >= 0
    fine_vals[land] = np.nan

    return FLON, FLAT, fine_vals

## 4. Plotting Functions

Each function below draws one panel of the multi-panel overview figure. They all follow the same signature pattern: data arrays, title string, and a Matplotlib `Axes` object.

### 4.1 Bathymetry

Uses filled contours (`contourf`) with the `ocean_r` colormap for depth and a tan fill for land. White isobath contour lines are labelled at 500 m intervals.

In [ ]:
def plot_bathymetry(lons, lats, elev, title, ax):
    """Filled-contour bathymetry with land mask and depth contours."""
    LON, LAT = np.meshgrid(lons, lats)
    bathy = np.where(elev < 0, elev, np.nan)

    d_min = np.nanmin(bathy)
    levels = np.linspace(d_min, 0, 30)
    cf = ax.contourf(LON, LAT, bathy, levels=levels,
                     cmap='ocean_r', extend='min')

    # Land
    ax.contourf(LON, LAT, np.where(elev >= 0, elev, np.nan),
                levels=[0, np.nanmax(elev) or 1], colors=['#c8b882'], zorder=2)
    ax.contour(LON, LAT, elev, levels=[0], colors='k', linewidths=0.8, zorder=3)

    # Labelled isobaths
    iso = np.arange(int(d_min / 1000) * 1000, 0, 500)
    if len(iso) > 1:
        cs = ax.contour(LON, LAT, bathy, levels=iso,
                        colors='white', linewidths=0.4, alpha=0.6)
        ax.clabel(cs, fontsize=6, fmt='%d m', inline=True)

    cbar = plt.colorbar(cf, ax=ax, shrink=0.82, pad=0.02)
    cbar.set_label('Depth (m)', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.15, ls='--')

### 4.2 Wave Climate (Mean $H_s$ + Direction Quivers)

The mean significant wave height field is interpolated to a fine grid and overlaid with **quiver arrows** showing the mean wave propagation direction on the original coarse grid.

In [ ]:
def plot_wave_climate(wave, title, ax, gebco=None):
    """Mean Hs contour map with mean wave direction quiver overlay."""
    hs = np.ma.filled(wave['mean_hs'], np.nan)
    d = np.ma.filled(wave['mean_dir'], np.nan)

    if gebco is not None:
        FLON, FLAT, hs_fine = upscale_and_mask(
            wave['lon'], wave['lat'], hs, gebco)
    else:
        FLON, FLAT = np.meshgrid(wave['lon'], wave['lat'])
        hs_fine = hs

    vmin = np.nanmin(hs_fine[hs_fine > 0]) if np.any(hs_fine > 0) else 0
    levels = np.linspace(vmin, np.nanmax(hs_fine), 25)
    cf = ax.contourf(FLON, FLAT, hs_fine, levels=levels, cmap='YlOrRd')
    cbar = plt.colorbar(cf, ax=ax, shrink=0.82, pad=0.02)
    cbar.set_label('Mean $H_s$ (m)', fontsize=10)

    LON_c, LAT_c = np.meshgrid(wave['lon'], wave['lat'])
    angle = np.deg2rad(270 - d)
    U, V = np.cos(angle), np.sin(angle)
    skip = max(1, len(wave['lat']) // 12)
    sl = (slice(None, None, skip), slice(None, None, skip))
    ax.quiver(LON_c[sl], LAT_c[sl], U[sl], V[sl],
              color='navy', scale=30, headwidth=3, headlength=4, alpha=0.85,
              zorder=4)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.15, ls='--')

### 4.3 Wave Power Density

Displays the time-averaged wave power density field $\bar{P}$ in kW/m using the `inferno` colormap. This is the key metric for wave energy resource assessment.

In [ ]:
def plot_wave_power(wave, title, ax, gebco=None):
    """Mean wave power density contour map."""
    P = np.ma.filled(wave['mean_P'], np.nan)

    if gebco is not None:
        FLON, FLAT, P_fine = upscale_and_mask(
            wave['lon'], wave['lat'], P, gebco)
    else:
        FLON, FLAT = np.meshgrid(wave['lon'], wave['lat'])
        P_fine = P

    vmin = np.nanmin(P_fine[P_fine > 0]) if np.any(P_fine > 0) else 0
    levels = np.linspace(vmin, np.nanmax(P_fine), 25)
    cf = ax.contourf(FLON, FLAT, P_fine, levels=levels, cmap='inferno')
    cbar = plt.colorbar(cf, ax=ax, shrink=0.82, pad=0.02)
    cbar.set_label('Wave Power (kW/m)', fontsize=10)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.15, ls='--')

### 4.4 Wind Climate (Mean Speed + Vectors)

Same interpolation approach as the wave maps, but showing mean 10-metre wind speed with wind-vector arrows.

In [ ]:
def plot_wind_climate(wind, title, ax, gebco=None):
    """Mean wind speed contour map with mean wind vectors."""
    spd = np.ma.filled(wind['mean_speed'], np.nan)

    if gebco is not None:
        FLON, FLAT, spd_fine = upscale_and_mask(
            wind['lon'], wind['lat'], spd, gebco)
    else:
        FLON, FLAT = np.meshgrid(wind['lon'], wind['lat'])
        spd_fine = spd

    levels = np.linspace(np.nanmin(spd_fine), np.nanmax(spd_fine), 25)
    cf = ax.contourf(FLON, FLAT, spd_fine, levels=levels, cmap='Blues')
    cbar = plt.colorbar(cf, ax=ax, shrink=0.82, pad=0.02)
    cbar.set_label('Mean Wind Speed (m/s)', fontsize=10)

    LON_c, LAT_c = np.meshgrid(wind['lon'], wind['lat'])
    u = np.ma.filled(wind['mean_u'], 0)
    v = np.ma.filled(wind['mean_v'], 0)
    skip = max(1, len(wind['lat']) // 14)
    sl = (slice(None, None, skip), slice(None, None, skip))
    ax.quiver(LON_c[sl], LAT_c[sl], u[sl], v[sl],
              color='darkred', scale=80, headwidth=3, alpha=0.8, zorder=4)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.15, ls='--')

### 4.5 Wind Rose

A **polar histogram** showing the frequency of wind from each of 16 compass directions, stacked by speed bins. This is the standard way to summarize wind climate for engineering applications.

In [ ]:
def plot_wind_rose(wind, title, ax):
    """Wind-rose histogram on a polar axes."""
    spd_flat = np.ma.filled(wind['speed'], np.nan).ravel()
    dir_flat = np.ma.filled(wind['direction'], np.nan).ravel()
    valid = ~np.isnan(spd_flat) & ~np.isnan(dir_flat)
    spd_flat = spd_flat[valid]
    dir_flat = dir_flat[valid]

    n_dir_bins = 16
    dir_edges = np.linspace(0, 360, n_dir_bins + 1)
    dir_centres = np.deg2rad((dir_edges[:-1] + dir_edges[1:]) / 2)
    bar_width = np.deg2rad(360 / n_dir_bins) * 0.9

    speed_bins = [0, 3, 6, 9, 12, 25]
    colors = ['#2166ac', '#67a9cf', '#fddbc7', '#ef8a62', '#b2182b']
    labels = ['0-3', '3-6', '6-9', '9-12', '>12 m/s']

    bottom = np.zeros(n_dir_bins)
    for k in range(len(speed_bins) - 1):
        mask = (spd_flat >= speed_bins[k]) & (spd_flat < speed_bins[k + 1])
        counts, _ = np.histogram(dir_flat[mask], bins=dir_edges)
        pct = 100 * counts / len(spd_flat)
        ax.bar(dir_centres, pct, width=bar_width, bottom=bottom,
               color=colors[k], edgecolor='white', linewidth=0.3,
               label=labels[k], zorder=3)
        bottom += pct

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=15)
    ax.legend(loc='lower left', bbox_to_anchor=(-0.15, -0.12),
              fontsize=8, ncol=3, title='Speed')

### 4.6 Wave Time Series

Plots domain-averaged $H_s$ and $T_p$ over the full simulation period, with:
- A **spatial min/max envelope** showing the range across the domain
- A **24-hour rolling mean** to highlight multi-day trends

This visualization helps identify storm events, seasonal variability, and data quality issues.

In [ ]:
def plot_wave_timeseries(wave, region_name, tag, out_dir):
    """Plot time series of domain-averaged Hs and Tp with min/max envelope."""
    import datetime

    times = wave['times']
    swh = np.ma.filled(wave['swh'], np.nan)
    tp  = np.ma.filled(wave['pp1d'], np.nan)

    py_times = np.array([datetime.datetime(t.year, t.month, t.day,
                                           t.hour, t.minute, t.second)
                         for t in times])

    hs_mean = np.nanmean(swh, axis=(1, 2))
    hs_min  = np.nanmin(swh, axis=(1, 2))
    hs_max  = np.nanmax(swh, axis=(1, 2))

    tp_mean = np.nanmean(tp, axis=(1, 2))
    tp_min  = np.nanmin(tp, axis=(1, 2))
    tp_max  = np.nanmax(tp, axis=(1, 2))

    def rolling(arr, w=24):
        kernel = np.ones(w) / w
        return np.convolve(arr, kernel, mode='same')

    hs_roll = rolling(hs_mean)
    tp_roll = rolling(tp_mean)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
    fig.suptitle(f'Wave Climate Time Series — {region_name}',
                 fontsize=15, fontweight='bold')

    ax1.fill_between(py_times, hs_min, hs_max,
                     color='#fddbc7', alpha=0.5, label='Spatial range')
    ax1.plot(py_times, hs_mean, color='#d6604d', lw=0.6,
             alpha=0.7, label='Domain mean')
    ax1.plot(py_times, hs_roll, color='#b2182b', lw=1.8,
             label='24-h rolling mean')
    ax1.set_ylabel('Significant Wave Height $H_s$ (m)', fontsize=11)
    ax1.legend(loc='upper right', fontsize=9, framealpha=0.9)
    ax1.grid(True, alpha=0.2, ls='--')
    ax1.set_ylim(bottom=0)

    ax2.fill_between(py_times, tp_min, tp_max,
                     color='#d1e5f0', alpha=0.5, label='Spatial range')
    ax2.plot(py_times, tp_mean, color='#4393c3', lw=0.6,
             alpha=0.7, label='Domain mean')
    ax2.plot(py_times, tp_roll, color='#2166ac', lw=1.8,
             label='24-h rolling mean')
    ax2.set_ylabel('Peak Wave Period $T_p$ (s)', fontsize=11)
    ax2.set_xlabel('Date', fontsize=11)
    ax2.legend(loc='upper right', fontsize=9, framealpha=0.9)
    ax2.grid(True, alpha=0.2, ls='--')
    ax2.set_ylim(bottom=0)

    ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax2.xaxis.set_minor_locator(mdates.DayLocator())
    fig.autofmt_xdate(rotation=30)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    p = os.path.join(out_dir, f'wave_timeseries_{tag}.png')
    fig.savefig(p, dpi=180, bbox_inches='tight')
    print(f"  Saved: {os.path.basename(p)}")
    return fig

## 5. Generate All Figures

The main loop below iterates over both study regions and produces:

1. A **2x2 overview figure** (bathymetry, wave height, wave power, wind speed)
2. A standalone **wind rose**
3. A **wave time series** panel
4. A **side-by-side bathymetry comparison** of both regions

All figures are saved as high-resolution PNGs.

In [ ]:
print("=" * 60)
print("Pre-Processing Visualization")
print("=" * 60)

for region_name, paths in REGIONS.items():
    tag = paths['tag']
    print(f"\n{'—'*60}")
    print(f"  Region: {region_name}")
    print(f"{'—'*60}")

    # --- Load data ---
    print("  Loading bathymetry...")
    g_lon, g_lat, g_elev = load_gebco(paths['gebco'])

    print("  Loading wave data...")
    wave = load_wave(paths['wave'])

    print("  Loading wind data...")
    wind = load_wind(paths['wind'])

    # --- Multi-panel figure (2x2) ---
    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    fig.suptitle(f'Pre-Processing Overview — {region_name}',
                 fontsize=16, fontweight='bold', y=0.98)

    gebco_tuple = (g_lon, g_lat, g_elev)

    plot_bathymetry(g_lon, g_lat, g_elev,
                    'Bathymetry (GEBCO)', axes[0, 0])

    print("  Interpolating wave data to fine grid...")
    plot_wave_climate(wave,
                      'Mean Wave Height & Direction (ERA5)', axes[0, 1],
                      gebco=gebco_tuple)

    plot_wave_power(wave,
                    'Mean Wave Power Density (ERA5)', axes[1, 0],
                    gebco=gebco_tuple)

    print("  Interpolating wind data to fine grid...")
    plot_wind_climate(wind,
                      'Mean Wind Speed & Vectors (ERA5)', axes[1, 1],
                      gebco=gebco_tuple)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    p = os.path.join(OUT, f'preprocessing_{tag}.png')
    fig.savefig(p, dpi=180, bbox_inches='tight')
    print(f"  Saved: {os.path.basename(p)}")
    plt.show()

    # --- Wind rose ---
    fig_wr, ax_wr = plt.subplots(
        subplot_kw={'projection': 'polar'}, figsize=(7, 7))
    plot_wind_rose(wind, f'Wind Rose — {region_name}', ax_wr)
    plt.tight_layout()
    p_wr = os.path.join(OUT, f'windrose_{tag}.png')
    fig_wr.savefig(p_wr, dpi=150, bbox_inches='tight')
    print(f"  Saved: {os.path.basename(p_wr)}")
    plt.show()

    # --- Wave time series ---
    print("  Plotting wave time series...")
    plot_wave_timeseries(wave, region_name, tag, OUT)
    plt.show()

## 6. Side-by-Side Bathymetry Comparison

Finally, we place both study regions side by side for direct visual comparison of their coastal morphology and depth profiles.

In [ ]:
print("Side-by-side bathymetry comparison")
fig_cmp, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(20, 10))
ca = REGIONS['California Bay']
om = REGIONS['Oman Coast']
l1, l2, e1 = load_gebco(ca['gebco'])
l3, l4, e2 = load_gebco(om['gebco'])
plot_bathymetry(l1, l2, e1, 'California Bay', ax_a)
plot_bathymetry(l3, l4, e2, 'Oman Coast', ax_b)
fig_cmp.suptitle('GEBCO Bathymetry — Study Regions',
                 fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
p_cmp = os.path.join(OUT, 'bathymetry_comparison.png')
fig_cmp.savefig(p_cmp, dpi=150, bbox_inches='tight')
print(f"  Saved: {os.path.basename(p_cmp)}")
plt.show()

print("\nDone.")